# 25 – Kịch bản chuyển động mô phỏng: kiểm toán nghiên cứu bộ pilot M2-R5

Notebook kiểm toán bộ pilot **SIMULATION** bốn lớp đã có (`ml/train/four_class_pilot_dataset.py`, `svmg-four-class-pilot-0.1.0`).
Không huấn luyện lại và không đổi ngưỡng nào: tín hiệu, nhãn v2 và cửa sổ đều lấy từ bộ mô phỏng có sẵn, bộ ngưỡng
labeling-v2 *candidate* đã đóng băng (`simulation/calibration/labeling_v2_candidate.json`) và các artifact pilot đã lưu.
Nhãn ở đây là **proxy theo kịch bản mô phỏng**, không phải ground truth ngoài thực tế.

**Chuẩn bị (checkout sạch):** chạy `.venv/bin/python -m ml.train.four_class_pilot_dataset` từ thư mục gốc repo để sinh
`ml/data/simulation_four_class_pilot/` (gitignored, khoảng 11 s). Notebook chạy được từ thư mục `ml/notebooks/` hoặc thư mục gốc.
Hình được ghi vào `ml/data/simulation_four_class_pilot/figures/` (gitignored); bản notebook trong repo không kèm output.

In [ ]:
import hashlib, json, platform, sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.train import four_class_pilot_dataset as pilot
from simulation.generator.config import DatasetSplit, LabelingVersion, ScenarioCode, SimulationConfig

DATA = ROOT / "ml/data/simulation_four_class_pilot"
FIG = DATA / "figures"
if not (DATA / "manifest.json").exists():
    raise FileNotFoundError("Chưa có dữ liệu pilot. Chạy từ thư mục gốc repo: .venv/bin/python -m ml.train.four_class_pilot_dataset")
FIG.mkdir(parents=True, exist_ok=True)
manifest = json.loads((DATA / "manifest.json").read_text())
windows = pd.read_csv(DATA / "windows.csv", keep_default_na=False)
assert manifest["dataset_sha256"] == hashlib.sha256((DATA / "windows.csv").read_bytes()).hexdigest(), "SHA của windows.csv không khớp manifest"
stale = [p for p, h in manifest["sources_sha256"].items() if hashlib.sha256((ROOT / p).read_bytes()).hexdigest() != h]
assert not stale, f"Mã nguồn đã đổi kể từ khi sinh pilot, hãy sinh lại dataset trước: {stale}"
CLASSES, SPLITS = list(manifest["class_mapping"]), ["Train", "Validation", "Test-G1"]
SPLIT_ENUM = {s.value: s for s in DatasetSplit}
sessions = pd.DataFrame(manifest["sessions"])
print("Thư mục làm việc của kernel:", Path.cwd().resolve().relative_to(ROOT) if Path.cwd().resolve() != ROOT else ".", "| interpreter:", Path(sys.executable).relative_to(ROOT) if ROOT in Path(sys.executable).parents else sys.executable)
print("Python", platform.python_version(), "|", manifest["dataset_version"], "|", manifest["label_policy"], "|", manifest["label_policy_status"])
print("SHA-256 dataset:", manifest["dataset_sha256"], "(khớp windows.csv; hash mã nguồn khớp manifest)")
print("Số session:", len(sessions), "| số cửa sổ:", len(windows), "| đủ điều kiện:", int((windows.label != "ABSTAIN").sum()))

## 1. Phân bố lớp và số session đóng góp theo split

Bảng 1: số cửa sổ đủ điều kiện; bảng 2: số session có ít nhất một cửa sổ đủ điều kiện; bảng 3: số session theo kế hoạch (cột theo lớp đích của kịch bản).

In [ ]:
counts = pd.DataFrame([{"split": s, "class": c, **v} for s, d in manifest["counts"].items() for c, v in d.items()])
win = counts.pivot(index="split", columns="class", values="windows").loc[SPLITS, CLASSES]
ses = counts.pivot(index="split", columns="class", values="sessions").loc[SPLITS, CLASSES]
planned = sessions.groupby(["split", "scenario"]).size().unstack().loc[SPLITS]
for table, caption in ((win, "Cửa sổ đủ điều kiện"), (ses, "Session đóng góp"), (planned.rename(columns=manifest["scenario_targets"]), "Session theo kế hoạch")):
    print(caption); display(table.rename_axis(index="Split", columns="Lớp"))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
win.plot.bar(ax=axes[0], rot=0, title="Số cửa sổ đủ điều kiện"); ses.plot.bar(ax=axes[1], rot=0, title="Số session đóng góp")
for ax in axes:
    ax.set_xlabel("Split"); ax.legend(title="Lớp", fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "01_class_distribution.png", dpi=90)

## 2. Tín hiệu roll và IMU đại diện (sinh lại bằng đúng seed)

Với mỗi lớp, chọn session Train có nhiều cửa sổ đủ điều kiện nhất và sinh lại bằng `ScenarioRunner` có sẵn. Các dòng pilot của session đó được dựng lại và phải trùng CSV đã lưu (kiểm tra tái lập). Trục IMU là hệ thân tàu FRD của bộ mô phỏng (nằm ngang: az = -g). Vùng cam: chuyển tiếp t0..t0+90 s; vạch xanh: điểm cuối các cửa sổ đủ điều kiện.

In [ ]:
runner, _ = pilot.make_runner()
_cache = {}
def regenerate(session):
    key = session["session_id"]
    if key not in _cache:
        _cache[key] = runner.generate(ScenarioCode(session["scenario"]), seed=int(session["seed"]),
                                      dataset_split=SPLIT_ENUM[session["split"]], labeling_version=LabelingVersion.V2)
    return _cache[key]

rep = (sessions[sessions.split == "Train"].sort_values(["scenario", "eligible_windows", "seed"], ascending=[True, False, True])
       .groupby("scenario").head(1).set_index("scenario"))
fig, axes = plt.subplots(4, 3, figsize=(15, 11), sharex=True)
for row, (scenario, s) in zip(axes, rep.iterrows()):
    r = regenerate(s.to_dict() | {"scenario": scenario})
    rows, _ = pilot.generate_session(runner, ScenarioCode(scenario), SPLIT_ENUM[s.split], int(s.seed))
    stored = windows[windows.session_id == s.session_id]
    assert [x["label"] for x in rows] == stored.label.tolist(), "Nhãn sinh lại khác CSV đã lưu"
    t = np.asarray(r.timestamps_s); imu = r.imu_samples
    series = [("roll [độ]", np.degrees(r.true_roll_rad)), ("gia tốc ay, az [m/s²]", None), ("con quay gx [độ/s]", np.degrees([x.gx for x in imu]))]
    for ax, (name, y) in zip(row, series):
        if y is None:
            ax.plot(t, [x.ay for x in imu], lw=.4, label="ay"); ax.plot(t, [x.az for x in imu], lw=.4, label="az"); ax.legend(fontsize=7)
        else:
            ax.plot(t, y, lw=.4)
        t0 = r.events[0]["t0_s"] if r.events else None
        if t0 is not None:
            ax.axvspan(t0, t0 + 90, color="orange", alpha=.25)
        for e in stored[stored.label != "ABSTAIN"].window_end_seconds:
            ax.axvline(float(e), color="green", lw=.3, alpha=.4)
        ax.set_title(f"{scenario} → {manifest['scenario_targets'][scenario]} · {name}", fontsize=9)
axes[-1, 1].set_xlabel("thời gian [s] (cam: chuyển tiếp t0..t0+90 s; xanh: điểm cuối cửa sổ đủ điều kiện)")
fig.tight_layout(); fig.savefig(FIG / "02_representative_signals.png", dpi=80)
print("Nhãn sinh lại của 4 session đại diện trùng khớp CSV đã lưu.")
display(rep[["session_id", "eligible_windows"]].rename_axis("Kịch bản").rename(columns={"eligible_windows": "Cửa sổ đủ điều kiện"}))

## 3. Khởi động, chuyển tiếp, ABSTAIN và cửa sổ đủ điều kiện theo thời gian

Mỗi hàng là một session, mỗi cột là một điểm cuối cửa sổ (bước 10 s). Lý do lấy từ CSV đã lưu: `CORE_FEATURE_INVALID` (Core C chưa đủ 180 s), `LABEL_WARMUP` (labeler v2 đang khởi động), `PRE_EVENT` (trước sự kiện), `TRANSITION_IN_WINDOW` (cửa sổ chứa đoạn chuyển tiếp), `ENDPOINT_LABEL_MISMATCH` (nhãn cuối cửa sổ khác nhãn đích), `PHENOTYPE_OVERLAP` (nhiều phenotype cùng bật), `ELIGIBLE` (đủ điều kiện).

In [ ]:
status = windows.label.where(windows.label == "ABSTAIN", "ELIGIBLE")
windows["status"] = np.where(status == "ELIGIBLE", "ELIGIBLE", windows.exclusion_reason)
order = ["CORE_FEATURE_INVALID", "LABEL_WARMUP", "PRE_EVENT", "TRANSITION_IN_WINDOW", "ENDPOINT_LABEL_MISMATCH", "PHENOTYPE_OVERLAP", "ELIGIBLE"]
grid = windows.pivot_table(index=["scenario", "split", "session_id"], columns="window_end_seconds", values="status", aggfunc="first")
codes = grid.apply(lambda c: c.map({k: i for i, k in enumerate(order)}))
fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(codes.to_numpy(float), aspect="auto", interpolation="nearest", cmap=plt.get_cmap("tab10", len(order)), vmin=-.5, vmax=len(order) - .5,
               extent=[grid.columns.min(), grid.columns.max(), len(grid) - .5, -.5])
ax.set_yticks(range(len(grid)), [f"{a} {b} {c[-4:]}" for a, b, c in grid.index], fontsize=6)
ax.set_xlabel("điểm cuối cửa sổ [s]"); ax.set_title("Trạng thái cửa sổ theo session (kịch bản, split, 4 số cuối seed)", fontsize=10)
cb = fig.colorbar(im, ticks=range(len(order))); cb.ax.set_yticklabels(order, fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "03_window_status_timeline.png", dpi=80)
display(pd.crosstab(windows.scenario, windows.status)[order].rename_axis(index="Kịch bản", columns="Trạng thái cửa sổ"))

## 4. Gán nhãn tại điểm cuối so với gán nhãn đồng nhất 180 s, và cửa sổ trộn trạng thái

Đồng nhất = mọi mẫu 20 Hz trong cửa sổ 180 s đều mang nhãn v2 đích (quy tắc kiểu G1). Tỉ lệ đích = phần mẫu trong cửa sổ có nhãn v2 bằng lớp được gán, tính trên session sinh lại.

In [ ]:
eligible = windows[windows.label != "ABSTAIN"].copy()
eligible["homogeneous"] = eligible.window_all_samples_agree.astype(str) == "True"
comp = eligible.groupby(["split", "label"]).homogeneous.agg(endpoint="size", homogeneous="sum").unstack("label").fillna(0).astype(int)
display(comp.loc[SPLITS].rename(columns={"endpoint": "Điểm cuối", "homogeneous": "Đồng nhất 180 s"}, level=0).rename_axis(index="Split", columns=["Quy tắc", "Lớp"]))
fractions, others = [], Counter()
for _, w in eligible.iterrows():
    r = regenerate(sessions.set_index("session_id").loc[w.session_id].to_dict() | {"session_id": w.session_id})
    end = int(round(w.window_end_seconds / 0.05)); labels = r.labels_v2[end - 3599:end + 1]
    fractions.append(sum(l == w.label for l in labels) / len(labels))
    others.update((w.label, l) for l in set(labels) - {w.label})
eligible["target_fraction"] = fractions
display(eligible.groupby("label").target_fraction.describe()[["count", "min", "25%", "50%", "mean"]]
        .rename(columns={"count": "số cửa sổ", "min": "nhỏ nhất", "25%": "phân vị 25%", "50%": "trung vị", "mean": "trung bình"}).rename_axis("Lớp"))
print("Nhãn v2 khác có mặt trong cửa sổ đủ điều kiện (lớp, nhãn khác): số cửa sổ =", dict(sorted(others.items())))
fig, ax = plt.subplots(figsize=(8, 3.4))
for c in CLASSES:
    ax.hist(eligible.loc[eligible.label == c, "target_fraction"], bins=np.linspace(0, 1, 21), histtype="step", label=c)
ax.set_xlabel("tỉ lệ mẫu trong cửa sổ mang nhãn v2 đích"); ax.set_ylabel("số cửa sổ đủ điều kiện")
ax.set_title("Mức trộn trạng thái trong cửa sổ đủ điều kiện", fontsize=10); ax.legend(title="Lớp", fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "04_mixed_state_fraction.png", dpi=90)

## 5. Vì sao AMPLITUDE_GROWTH thưa

Với mỗi session S3: tỉ lệ thời gian theo nhãn v2 sau chuyển tiếp (t ≥ t0+90 s), thời điểm đầu tiên có growth / HIGH_ROLL (tính từ t0), đợt growth liên tục dài nhất, nhãn v2 tại các điểm cuối bị loại vì lệch nhãn, và số cửa sổ đủ điều kiện. Đường nét đứt h = 4ζ là biên mất ổn định Mathieu tuyến tính khi tắt dần nhỏ cho dạng phương trình này, chỉ để tham chiếu (bộ mô phỏng có thêm độ cứng bậc ba và kích động ngẫu nhiên).

In [ ]:
rows = []
for _, s in sessions[sessions.scenario == "S3"].iterrows():
    r = regenerate(s); t = np.asarray(r.timestamps_s); t0 = r.events[0]["t0_s"]; post = t >= t0 + 90
    labels = np.array([l or "WARMUP" for l in r.labels_v2]); occ = Counter(labels[post])
    growth = labels == "AMPLITUDE_GROWTH"; runs, cur = [], 0
    for g in growth:
        cur = cur + 1 if g else 0; runs.append(cur)
    first = lambda name: next((round(x - t0) for x, l in zip(t, labels) if l == name and x >= t0), None)
    p = s.sampled_parameters
    ex = windows[(windows.session_id == s.session_id) & (windows.exclusion_reason == "ENDPOINT_LABEL_MISMATCH")]
    endpoint = Counter(r.labels_v2[int(round(e / 0.05))] for e in ex.window_end_seconds)
    rows.append({"split": s.split, "seed": s.seed, "h": round(p["parametric_excitation_amplitude"], 3), "zeta": round(p["damping_ratio"], 3),
                 "T0_s": round(p["natural_period_s"], 1), **{f"post_{k}": round(occ[k] / post.sum(), 3) for k in ("NORMAL", "HIGH_ROLL", "AMPLITUDE_GROWTH")},
                 "first_growth_s": first("AMPLITUDE_GROWTH"), "first_high_roll_s": first("HIGH_ROLL"), "longest_growth_run_s": max(runs) * 0.05,
                 "mismatch_endpoint_labels": dict(endpoint), "eligible": s.eligible_windows})
s3 = pd.DataFrame(rows)
display(s3.rename(columns={"post_NORMAL": "tỉ lệ NORMAL sau chuyển tiếp", "post_HIGH_ROLL": "tỉ lệ HIGH_ROLL sau chuyển tiếp",
                           "post_AMPLITUDE_GROWTH": "tỉ lệ AMPLITUDE_GROWTH sau chuyển tiếp", "first_growth_s": "growth đầu tiên [s]",
                           "first_high_roll_s": "HIGH_ROLL đầu tiên [s]", "longest_growth_run_s": "đợt growth dài nhất [s]",
                           "mismatch_endpoint_labels": "nhãn tại điểm cuối bị loại", "eligible": "cửa sổ đủ điều kiện"}))
print("Session S3 có cửa sổ growth đủ điều kiện:", int((s3.eligible > 0).sum()), "/", len(s3), "| theo split:", s3.groupby("split").eligible.sum().to_dict())
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(s3.zeta, s3.h, c=np.where(s3.eligible > 0, "tab:green", "tab:red"))
z = np.linspace(0, s3.zeta.max() * 1.1, 10); ax.plot(z, 4 * z, "k--", lw=.8, label="h = 4ζ (tham chiếu)")
ax.set_xlabel("hệ số tắt dần ζ"); ax.set_ylabel("biên độ kích thích tham số h"); ax.legend(fontsize=7)
ax.set_title("Session S3 (xanh: ≥1 cửa sổ growth đủ điều kiện; đỏ: không có)", fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "05_growth_parameters.png", dpi=90)

## 6. Độ đa dạng seed và độ phủ tham số

Tham số lấy từ manifest. Ô thứ hai sinh lại kế hoạch S1 với seed tuần tự (cách G1 đã dùng) so với seed băm của pilot, chỉ dùng runner có sẵn.

In [ ]:
params = pd.DataFrame([{"split": s["split"], "scenario": s["scenario"], **s["sampled_parameters"]} for s in manifest["sessions"]])
domains = SimulationConfig().dataset.domains
display(params.groupby("split").agg(t0_min=("t0_s", "min"), t0_max=("t0_s", "max"), T0_min=("natural_period_s", "min"),
        T0_max=("natural_period_s", "max"), zeta_min=("damping_ratio", "min"), zeta_max=("damping_ratio", "max")).loc[SPLITS].rename_axis("Split"))
display(pd.crosstab(params.scenario, params.baseline_sea_state).rename_axis(index="Kịch bản", columns="Sea state nền"))
for split, name in (("Train", "train"), ("Validation", "validation"), ("Test-G1", "test_g1")):
    dom, sub = getattr(domains, name).parameters, params[params.split == split]
    for key in ("natural_period_s", "damping_ratio", "t0_s"):
        vals = sub[key].dropna(); assert vals.between(dom[key].minimum, dom[key].maximum).all(), (split, key)
print("Mọi giá trị T0/ζ/t0 đã lấy mẫu đều nằm trong miền tham số đã thống nhất của từng split;",
      "số thời điểm sự kiện t0 khác nhau:", params.t0_s.dropna().round().nunique(), "/", params.t0_s.notna().sum())

In [ ]:
def plan_of(seed):
    md = runner.generate(ScenarioCode.S1, seed=seed, dataset_split=DatasetSplit.TRAIN, labeling_version=LabelingVersion.V2).metadata
    return md["sampled_parameters"]["t0_s"], md["sampled_parameters"]["baseline_sea_state"]
seq = [plan_of(manifest["seed_base"] + i + 1) for i in range(12)]
hashed = [plan_of(pilot.session_seed(i, manifest["seed_base"])) for i in range(12)]
for name, plans in (("seed tuần tự", seq), ("seed băm", hashed)):
    print(f"{name:12s} số t0 khác nhau={len({round(t) for t, _ in plans})}/12, khoảng t0={min(t for t, _ in plans):.1f}-{max(t for t, _ in plans):.1f} s,",
          "sea state:", dict(Counter(ss for _, ss in plans)))
g1 = ROOT / "ml/data/simulation_rf_g1/manifest.json"
if g1.exists():
    g1_seeds = [s["seed"] for s in json.loads(g1.read_text())["sessions"] if s["scenario"] == "S1"]
    g1_plans = [plan_of(seed) for seed in g1_seeds]
    print("Seed S1 của G1:", g1_seeds[0], "...", "| số t0 khác nhau", len({round(t) for t, _ in g1_plans}), "/", len(g1_plans),
          "| sea state:", dict(Counter(ss for _, ss in g1_plans)))
else:
    print("Không có manifest G1 trên máy; phép so sánh G1: NOT_RUN")

## 7. Nhãn kịch bản mô phỏng khác với ground truth ngoài thực tế

* Lớp được gán theo **kịch bản đã sinh ra session**, rồi xác nhận bằng ngưỡng labeling-v2 *candidate* tính từ chính chuỗi roll
  mô phỏng đó. Nhãn mô tả điều bộ mô phỏng được cấu hình để làm, không phải trạng thái tàu được quan sát hay chuyên gia xác
  minh; `NORMAL` nghĩa là cấu hình biển lặng S0, không phải vùng vận hành bình thường đã được kiểm chứng.
* Ngưỡng được hiệu chỉnh trên các session miền Validation của cùng bộ mô phỏng (`calibration_status=candidate`).
* Không cửa sổ Hohai, Seafar hay Southampton nào được gán nhãn bốn lớp ở đây; dữ liệu thật vẫn chưa có nhãn cho ontology này.

## 8. Giới hạn khoa học

* Nhãn và đặc trưng cùng sinh từ một chuỗi roll mô phỏng; nhãn dạng quy tắc có thể làm việc phân lớp trông dễ (xem M2-R3).
* Bộ mô phỏng chỉ có roll, nhiễu/bias IMU chưa đặt: ba đặc trưng pitch bằng đúng 0; MPU6050 thật sẽ khác.
* Profile kích động sea state chưa được hiệu chỉnh; Test-G2 (JONSWAP) chưa được triển khai.
* Nhãn cửa sổ theo chính sách điểm cuối đang PROPOSED; phương án đồng nhất và số cửa sổ trộn trạng thái được trình bày ở trên.
* Pilot nhỏ (40 session); số lượng không phù hợp cho metric cuối cùng hay tuyên bố độ chính xác ngoài thực tế.